In [19]:
import pandas as pd


In [20]:
path = "../data/interim/evaluations/neo4j-2024v1/llama3.1-8b/test.csv"
df = pd.read_csv(path)
print(df.info())
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2461 entries, 0 to 2460
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   df.index            2461 non-null   int64  
 1   jaccard_similarity  1988 non-null   float64
 2   google_bleu         2461 non-null   float64
 3   rouge_l_f1          2461 non-null   float64
dtypes: float64(3), int64(1)
memory usage: 77.0 KB
None
   df.index  jaccard_similarity  google_bleu  rouge_l_f1
0      4265                 1.0     0.444444    0.888889
1      4162                 NaN     0.469231    0.790698
2      2356                 0.0     0.688679    0.709677
3      4204                 1.0     0.530303    0.900000
4      2371                 NaN     0.412281    0.595745


# Loading Data

In [21]:
models = ["gpt-4.1", "gemma-2-9b-it", "llama3.1-8b"]
metrics = ['jaccard_similarity', 'exact_match', 'google_bleu', 'rouge_l_f1']
results = dict()
for model in models:
    df = pd.read_csv(f"../data/interim/evaluations/neo4j-2024v1/{model}/test.csv")
    na_to_0_df = df.fillna({metric: 0 for metric in metrics})
    na_to_0_df['exact_match'] = (na_to_0_df['jaccard_similarity'] == 1.0).apply(int)
    results[model] = {
        metric: (na_to_0_df[metric].mean(), na_to_0_df[metric].std()) 
        for metric in metrics
    }

In [24]:
# print a comparison table
head_str = [f"{'Model':<20}"] + [f"{metric.replace('_', ' ').title():<20} " for metric in metrics]
print("".join(head_str))
for model in models:
    row_str = [f"{model} "] + [f"{results[model][metric][0]:.4f} ± {results[model][metric][1]:.4f} " for metric in metrics]
    row_str = [f"{row:<20}" for row in row_str]
    print("".join(row_str))

Model               Jaccard Similarity   Exact Match          Google Bleu          Rouge L F1           
gpt-4.1             0.3561 ± 0.4707     0.3364 ± 0.4726     0.5339 ± 0.2632     0.6294 ± 0.2708     
gemma-2-9b-it       0.3523 ± 0.4698     0.3348 ± 0.4720     0.6400 ± 0.2054     0.7364 ± 0.1774     
llama3.1-8b         0.2253 ± 0.4133     0.2166 ± 0.4120     0.5906 ± 0.2074     0.6898 ± 0.1852     
